# Man-in-the-Middle — Spoofing the GCS's Proximity Guard

Same scenario as `5-GCS_intervention_proximity.ipynb`: a single drone flies an
AUTO mission across the city, on a leg that runs straight into the
`radio_tower_intervention` standing on the corridor. Normally the GCS watches
the drone's position and acts as a **guard**: while the drone is within
`radius` of the tower it takes over (GUIDED + a guided detour), keeping it
from ever reaching the obstacle.

Here an attacker has man-in-the-middled the drone's telemetry link to the GCS.
Rather than spoofing from the very start, the MITM watches the drone's real
position and only engages **once it is genuinely closing in on the tower** —
a little before the real guard's own trigger radius — and from then on keeps
reporting the exact position it saw at that instant, frozen, forever. The
`ProximityTrigger` reads its "where is the drone" straight from that stream,
so it never sees the vehicle actually enter the guarded zone and never
engages. The drone itself is untouched: Logic still reads its own SITL
connection directly, so the real AUTO mission flies on exactly as planned;
only what the GCS is *told* changes.

In [ ]:
from simulator import Oracle, Simulator
from simulator.config import Color, Model
from simulator.entities import Intervention, ProximityTrigger, SimGCS, SimVehicle
from simulator.helpers import clean
from simulator.helpers.coordinates import ENU, ENUPose, GRAPose
from simulator.helpers.processes import SimProcess
from simulator.planner import AutoPlan, InterventionPlan
from simulator.runtime.mitm import CloakPositionStrategy
from simulator.visualizer import Gazebo, GazMarker

clean()

## Origin and home positions

In [ ]:
gra_origin = GRAPose(lat=-35.3633280, lon=149.1652241, alt=0, heading=0)
enu_origin = ENUPose(x=0, y=0, z=gra_origin.alt, heading=gra_origin.heading)

home = ENUPose(0, 0, 0, 0)

## Waypoints

In [ ]:
cruise_alt = 15.0
home_wp = ENU(x=0, y=0, z=0)
headup_wp = ENU(x=0, y=0, z=cruise_alt)
end_wp = ENU(x=-85, y=56, z=cruise_alt)
mission_wps = [home_wp, headup_wp, end_wp]

## Vehicle

In [ ]:
sysid = 1
model = Model.IRIS

plan = AutoPlan.from_relative_path(
    name="north_mission",
    sysid=sysid,
    gra_origin=gra_origin,
    relative_home=home,
    relative_path=mission_wps,
    navigation_speed=3.0,
    firmware=model.firmware,
)

vehicle = SimVehicle.from_relative(
    sysid=sysid,
    plan=plan,
    color=Color.BLUE,
    enu_origin=enu_origin,
    relative_home=home,
    relative_path=mission_wps,
    model=model,
)


## GCS

In [ ]:
gcs_color = Color.BLUE
gcs = SimGCS(name=f"{gcs_color.name}_{gcs_color.emoji}", record_positions=True)
gcs.own(vehicle)

### Intervention

`gcs.intervene` requires the GCS to already monitor the vehicle.

A **`ProximityTrigger`** is a *guard*: reversible, so the GCS hands control back
when the drone leaves the zone. Unlike `5-GCS_intervention_proximity.ipynb`,
this one is left **enabled** below — the guard is real and would work, if the
GCS could actually see the drone coming.

* The `radio_tower_intervention` in the world sits on the mission corridor: the straight leg (0,0) -> (-85,56) passes through (-47, 31), so the AUTO mission flies the drone straight into it.
* Guard geometry. `radius` is where the GCS takes over; `release_radius` is where it hands back (hysteresis, so it does not chatter at the boundary).
* Where the GCS steers the drone while it holds control: offset perpendicular to the corridor, far enough that once released the resumed mission leg no longer re-enters the guarded zone (otherwise AUTO would fly straight back in).

In [ ]:
obstacle = ENU(x=-47, y=30, z=cruise_alt)
guard_radius = 25.0  # meters
release_radius = 45.0  # meters
detour = ENU(x=-22, y=68, z=cruise_alt)

print(
    f"Obstacle at {obstacle.short()}, guard {guard_radius:.0f} m, release "
    f"{release_radius:.0f} m"
)

In [ ]:
gcs.intervene(
    vehicle,
    Intervention(
        trigger=ProximityTrigger(
            near=obstacle,
            radius=guard_radius,
            release_radius=release_radius,
        ),
        plan=InterventionPlan.from_relative_path(
            relative_path=[detour],
            enu_origin=enu_origin,
            relative_home=home,
            firmware=model.firmware,
            land=False,
        ),
        firmware=model.firmware,
    ),
)

## MITM: cloak the drone once it nears the tower

Spoofing from the very first telemetry sample would be needlessly conspicuous
and isn't how a real attacker would time it. Instead the MITM reuses the
**exact same `Trigger`** a GCS `Intervention` would use — a `ProximityTrigger`
around the same `obstacle`, just with a wider `radius` so it fires before the
real guard ever would — to decide when the vehicle is genuinely closing in.
While the trigger doesn't hold, telemetry passes through unmodified; the
instant it first holds, every `GLOBAL_POSITION_INT` afterward is replaced
with the exact report the vehicle sent at that instant.

From the GCS's point of view the drone approaches... and then simply stops
moving, forever, just outside the guarded zone. The real `ProximityTrigger`
reads its position from that same frozen stream, so it never crosses its own
`radius` and never engages. The drone's own flight controller and Logic never
see any of this — only what the GCS is told changes.

In [ ]:
mitm_trigger_radius = (
    guard_radius + 10.0
)  # meters; must clear the guard's own radius first

# Reuses the exact same ProximityTrigger the real guard uses (same `near`),
# just with a wider radius, so this fires before the real guard ever would.
vehicle.mitm = CloakPositionStrategy(
    trigger=ProximityTrigger(near=obstacle, radius=mitm_trigger_radius),
)
print(
    f"MITM freezes the GCS's view once within {mitm_trigger_radius:.0f} m of the obstacle"
)

## Oracle

In [ ]:
orac = Oracle()
orac.add_gcs(gcs)

## Visualizer

In [ ]:
gaz = Gazebo(
    gra_origin,
    world_path="simulator/visualizer/gazebo/worlds/small_city_intervention.world",
)
# The zone the GCS guards: inside the red sphere it would take over, if it
# could see the drone enter it.
# guard_marker = GazMarker(
#     name="guard_zone",
#     group="guard",
#     pos=obstacle,
#     color=Color.RED,
#     radius=guard_radius,
#     alpha=0.6,
# )
# # Where it would steer the drone while holding control.
detour_marker = GazMarker(
    name="detour",
    group="targets",
    pos=detour,
    color=Color.RED,
)
for marker in [detour_marker]:  # detour_marker is now included
    gaz.markers.append(marker)

## Simulator

In [ ]:
simulator = Simulator(
    oracle=orac,
    visualizer=gaz,
    verbose=1,
    terminals=[SimProcess.LOGIC, SimProcess.GCS, SimProcess.MITM],
)

simulator.preview()


In [ ]:
simulator.run()


## Verifying the attack

- `simulator/logs/mitm/mitm_1.log` should show `MITM cloak: trigger fired -
  freezing the GCS's view here` once, when the drone reaches
  `mitm_trigger_radius`, and then no proximity engagement at all — the GCS
  never sees the drone within `radius` of the tower afterward.
- **Oracle view** (`oracle=True`) is the *real* flight, from the vehicle's own
  Remote ID broadcast — a separate channel the MITM never touches. It should
  show the AUTO mission flying straight through the tower's position,
  unimpeded, following the planned waypoints.
- **GCS view** (`gcss=[gcs.name]`) is what the GCS itself recorded from its
  own (cloaked) telemetry — it should track the real flight normally up to
  `mitm_trigger_radius`, then freeze at that last real point for the rest of
  the run, just outside the guarded zone.

In [ ]:
orac.plot_trajectories(oracle=True, waypoints=True, azim=-70, elev=15);


In [ ]:
orac.plot_trajectories(
    oracle=False, gcss=[gcs.name], waypoints=True, azim=-70, elev=15
);
